<a href="https://colab.research.google.com/github/anubhavtiwari-cyber/ai-ml-internship-maincrafts/blob/main-maincraft/task1_ml_linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 Task 1: Linear Regression — California Housing Price Predictor
**AI/ML Internship — Maincrafts Technology**

**Objective:** Train a Linear Regression model to predict median house prices using the California Housing dataset.

**Steps Covered:**
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Preprocessing & Train/Test Split
4. Model Training
5. Evaluation (MAE, RMSE, R²)
6. Visualization
7. Save Model

## 📦 Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib

# Plot style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn joblib


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 📂 Step 2: Load Dataset

In [ ]:
# Load California Housing dataset
data = fetch_california_housing(as_frame=True)

# Combine features + target into one DataFrame
df = pd.concat([data.data, data.target.rename('MedHouseVal')], axis=1)

print(f'Dataset Shape: {df.shape}')
print(f'\nFeatures: {list(data.feature_names)}')
print(f'Target: MedHouseVal (Median House Value in $100,000s)')
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
df.info()

In [ ]:
# Statistical summary
print('=== Statistical Summary ===')
df.describe().round(2)

In [ ]:
# Check for missing values
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Median House Value', fontsize=14)
axes[0].set_xlabel('MedHouseVal ($100,000s)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['MedHouseVal'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Boxplot of Median House Value', fontsize=14)
axes[1].set_ylabel('MedHouseVal ($100,000s)')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Target distribution plotted')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Correlation heatmap plotted')
print('\n🔍 Top features correlated with MedHouseVal:')
print(corr_matrix['MedHouseVal'].sort_values(ascending=False).round(3))

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=40, color='teal', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribution: {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Feature distributions plotted')

In [ ]:
# Scatter plots: top 4 features vs target
top_features = corr_matrix['MedHouseVal'].abs().sort_values(ascending=False)[1:5].index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['MedHouseVal'], alpha=0.1, color='steelblue', s=5)
    axes[i].set_xlabel(feat, fontsize=12)
    axes[i].set_ylabel('MedHouseVal', fontsize=12)
    axes[i].set_title(f'{feat} vs MedHouseVal', fontsize=13)

plt.suptitle('Top Features vs Target Variable', fontsize=16)
plt.tight_layout()
plt.savefig('feature_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Feature vs Target scatter plots done')

## ⚙️ Step 4: Preprocessing & Train/Test Split

In [ ]:
# Separate features and target
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set size : {X_train.shape[0]} samples')
print(f'Test set size     : {X_test.shape[0]} samples')

# Feature scaling (StandardScaler)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('\n✅ Data split and scaled successfully!')

## 🤖 Step 5: Train Linear Regression Model

In [ ]:
# Initialize and train the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print('✅ Model trained!')
print(f'\nIntercept : {model.intercept_:.4f}')
print('\nFeature Coefficients:')
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
coef_df = coef_df.sort_values('Coefficient', ascending=False)
print(coef_df.to_string(index=False))

In [ ]:
# Feature importance (coefficient) bar plot
plt.figure(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Feature Coefficients (Linear Regression)', fontsize=14)
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('feature_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 📏 Step 6: Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test_scaled)

# Metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2   = r2_score(y_test, y_pred)

print('='*45)
print('       📊 MODEL EVALUATION RESULTS')
print('='*45)
print(f'  MAE  (Mean Absolute Error)  : {mae:.4f}')
print(f'  RMSE (Root Mean Sq. Error)  : {rmse:.4f}')
print(f'  R²   (R-Squared Score)      : {r2:.4f}')
print('='*45)
print(f'\n✅ Model explains {r2*100:.1f}% variance in house prices')
print(f'✅ Average prediction error: ${mae*100000:,.0f}')

## 📊 Step 7: Visualization of Results

In [ ]:
# Plot 1: Actual vs Predicted
plt.figure(figsize=(10, 7))
plt.scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=10, label='Predictions')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual Values (MedHouseVal)', fontsize=13)
plt.ylabel('Predicted Values (MedHouseVal)', fontsize=13)
plt.title(f'Actual vs Predicted House Values\nR² = {r2:.4f} | RMSE = {rmse:.4f}', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Actual vs Predicted plot saved')

In [ ]:
# Plot 2: Residuals plot
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.3, color='darkorange', s=10)
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Values', fontsize=12)
axes[0].set_ylabel('Residuals', fontsize=12)
axes[0].set_title('Residuals vs Predicted', fontsize=13)

# Residual distribution
axes[1].hist(residuals, bins=60, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual Value', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=13)

plt.suptitle('Residual Analysis', fontsize=16)
plt.tight_layout()
plt.savefig('residuals_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Residuals plot saved')

## 💾 Step 8: Save the Model

In [ ]:
# Save model and scaler using joblib
joblib.dump(model, 'linear_regression_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('✅ Model saved as: linear_regression_model.pkl')
print('✅ Scaler saved as: scaler.pkl')

## 🔮 Step 9: Predict on New Input (Optional UI)

In [ ]:
# Predict on a new custom sample
# Feature order: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude

new_sample = pd.DataFrame([{
    'MedInc'      : 5.0,    # Median income in block ($10,000s)
    'HouseAge'    : 20.0,   # Median house age (years)
    'AveRooms'    : 6.0,    # Average rooms per house
    'AveBedrms'   : 1.1,    # Average bedrooms per house
    'Population'  : 1200.0, # Block population
    'AveOccup'    : 3.0,    # Average occupants per house
    'Latitude'    : 34.0,   # Latitude
    'Longitude'   : -118.0  # Longitude (Los Angeles area)
}])

new_sample_scaled = scaler.transform(new_sample)
predicted_price   = model.predict(new_sample_scaled)[0]

print(f'🏠 Predicted House Value : ${predicted_price * 100000:,.0f}')
print(f'   (i.e., {predicted_price:.3f} in $100,000 units)')

## 📝 Step 10: Summary & Improvement Ideas

In [ ]:
print('='*55)
print('           📋 FINAL PROJECT SUMMARY')
print('='*55)
print(f'  Dataset     : California Housing (sklearn)')
print(f'  Samples     : {len(df):,}')
print(f'  Features    : {X.shape[1]}')
print(f'  Train/Test  : 80% / 20%')
print(f'  Model       : Linear Regression')
print(f'  MAE         : {mae:.4f}')
print(f'  RMSE        : {rmse:.4f}')
print(f'  R² Score    : {r2:.4f}')
print('='*55)
print()
print('💡 Improvement Ideas:')
print('  1. Try Ridge / Lasso Regression (regularization)')
print('  2. Use Random Forest or XGBoost for better R²')
print('  3. Feature engineering (e.g., rooms per person)')
print('  4. Remove outliers (cap MedHouseVal at 5.0)')
print('  5. Cross-validation for more robust evaluation')
print('  6. Hyperparameter tuning with GridSearchCV')